In [1]:
import pandas as pd
import numpy as np

In [2]:
filename = 'diabetes_dataset.csv'
df = pd.read_csv(filename)
df.head(1)

,age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1


In [3]:
df['systolic_diastolic_ratio'] = df['systolic_bp'] / df['diastolic_bp']
df['hdl_ldl_ratio'] = df['hdl_cholesterol'] / df['ldl_cholesterol']
df['glucose_after_eating_to_fasting_ratio'] = df['glucose_postprandial'] / df['glucose_fasting']
df ['bmi_to_age_ratio'] = df['bmi'] / df['age']
df['bmi_cholesterol_ratio'] = df['bmi'] / df['cholesterol_total']

In [4]:
input_cols = ['age', 'gender','smoking_status', 'alcohol_consumption_per_week',
       'sleep_hours_per_day' , 'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history', 'bmi','cholesterol_total' , 'triglycerides', 'insulin_level', 'hba1c', 
       'physical_activity_minutes_per_week', 'diet_score',
       'systolic_diastolic_ratio', 'hdl_ldl_ratio',
       'glucose_after_eating_to_fasting_ratio', 'bmi_to_age_ratio',
       'bmi_cholesterol_ratio']

target_col = 'diabetes_stage'

In [5]:
numerical_cols = df[input_cols].select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = df[input_cols].select_dtypes(include=['object']).columns.tolist()

In [6]:
from sklearn.model_selection import train_test_split
X = df[input_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)
X_train.shape, X_test.shape

((90000, 20), (10000, 20))

In [7]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train[cat_cols])
ohe.categories_

[array(['Female', 'Male', 'Other'], dtype=object),
 array(['Current', 'Former', 'Never'], dtype=object)]

In [8]:
X_train_ohe = ohe.transform(X_train[cat_cols])
X_test_ohe = ohe.transform(X_test[cat_cols])

In [9]:
input_cols_final = numerical_cols + ohe.get_feature_names_out(cat_cols).tolist()

In [10]:
X_train_final = pd.DataFrame(X_train_ohe, columns=ohe.get_feature_names_out(cat_cols), index=X_train.index)
X_train_final[numerical_cols] = X_train[numerical_cols]
X_test_final = pd.DataFrame(X_test_ohe, columns=ohe.get_feature_names_out(cat_cols), index=X_test.index)
X_test_final[numerical_cols] = X_test[numerical_cols]
X_train_final.head()

,gender_Female,gender_Male,gender_Other,smoking_status_Current,smoking_status_Former,smoking_status_Never,age,alcohol_consumption_per_week,sleep_hours_per_day,family_history_diabetes,...,triglycerides,insulin_level,hba1c,physical_activity_minutes_per_week,diet_score,systolic_diastolic_ratio,hdl_ldl_ratio,glucose_after_eating_to_fasting_ratio,bmi_to_age_ratio,bmi_cholesterol_ratio
51994,1.0,0.0,0.0,0.0,0.0,1.0,67,2,8.9,0,...,93,14.90,5.86,34,5.0,2.015385,0.405405,1.731183,0.434328,0.125431
77540,1.0,0.0,0.0,0.0,0.0,1.0,39,1,7.1,0,...,129,3.49,6.82,95,8.6,1.739130,0.420168,1.542857,0.607692,0.119697
16382,1.0,0.0,0.0,1.0,0.0,0.0,36,5,6.1,0,...,167,15.47,7.10,419,5.0,1.585714,0.650000,1.596491,0.691667,0.141477
83439,1.0,0.0,0.0,0.0,0.0,1.0,62,1,7.2,0,...,112,13.97,7.50,55,2.6,1.686047,0.388350,1.458015,0.574194,0.189362
61618,1.0,0.0,0.0,1.0,0.0,0.0,31,3,6.8,0,...,98,7.59,7.40,138,6.1,1.541667,0.343511,1.660714,1.054839,0.151389


In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

In [13]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score

C:\Users\KUSHANKUR\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [17]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'num_leaves': trial.suggest_int('num_leaves', 15, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42
    }

    model = LGBMClassifier(**params)
    score = cross_val_score(model, X_train_final, y_train_enc, cv=cv, scoring='accuracy', n_jobs=-1).mean()
    return score  # Optuna maximizes this by default if direction='maximize'


In [18]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

# Best parameters and accuracy
print("Best parameters:", study.best_params)
print("Best CV accuracy:", study.best_value)

[I 2025-10-26 11:37:22,641] A new study created in memory with name: no-name-9803e05a-1ead-4de9-b27c-9546174f04b5
Best trial: 0. Best value: 0.883067:   2%|▎         | 1/40 [01:15<48:57, 75.31s/it]

[I 2025-10-26 11:38:37,952] Trial 0 finished with value: 0.8830666666666666 and parameters: {'n_estimators': 851, 'learning_rate': 0.03599899994059888, 'max_depth': 5, 'num_leaves': 65, 'subsample': 0.5137225354508326, 'colsample_bytree': 0.8689424579278762}. Best is trial 0 with value: 0.8830666666666666.


Best trial: 1. Best value: 0.883656:   5%|▌         | 2/40 [02:51<55:34, 87.75s/it]

[I 2025-10-26 11:40:14,404] Trial 1 finished with value: 0.8836555555555556 and parameters: {'n_estimators': 819, 'learning_rate': 0.022174246575108768, 'max_depth': 11, 'num_leaves': 64, 'subsample': 0.8911620219800837, 'colsample_bytree': 0.7276518852825433}. Best is trial 1 with value: 0.8836555555555556.


Best trial: 2. Best value: 0.885178:   8%|▊         | 3/40 [03:49<45:37, 73.99s/it]

...

[I 2025-10-26 12:09:47,696] Trial 37 finished with value: 0.839911111111111 and parameters: {'n_estimators': 460, 'learning_rate': 0.001315119151734218, 'max_depth': 10, 'num_leaves': 45, 'subsample': 0.6765748218933056, 'colsample_bytree': 0.7078255814086208}. Best is trial 20 with value: 0.8855222222222222.


Best trial: 20. Best value: 0.885522:  98%|█████████▊| 39/40 [33:22<00:54, 54.21s/it]

[I 2025-10-26 12:10:45,485] Trial 38 finished with value: 0.8853 and parameters: {'n_estimators': 573, 'learning_rate': 0.003590421745097545, 'max_depth': 12, 'num_leaves': 36, 'subsample': 0.6256158652382773, 'colsample_bytree': 0.8564140034761568}. Best is trial 20 with value: 0.8855222222222222.


Best trial: 20. Best value: 0.885522: 100%|██████████| 40/40 [34:11<00:00, 51.29s/it]

[I 2025-10-26 12:11:34,250] Trial 39 finished with value: 0.8851888888888888 and parameters: {'n_estimators': 669, 'learning_rate': 0.020098275368129595, 'max_depth': 11, 'num_leaves': 15, 'subsample': 0.570163967931135, 'colsample_bytree': 0.7978368029002432}. Best is trial 20 with value: 0.8855222222222222.
Best parameters: {'n_estimators': 479, 'learning_rate': 0.005082590397179811, 'max_depth': 12, 'num_leaves': 26, 'subsample': 0.6568567589536585, 'colsample_bytree': 0.9325224800427339}
Best CV accuracy: 0.8855222222222222


In [19]:
model = LGBMClassifier(
    n_estimators = 479,
    learning_rate = 0.005082590397179811,
    max_depth = 12,
    num_leaves = 26,
    subsample = 0.6568567589536585,
    colsample_bytree = 0.9325224800427339
)
model.fit(X_train_final, y_train_enc)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.011085 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3009
[LightGBM] [Info] Number of data points in the train set: 90000, number of used features: 24
[LightGBM] [Info] Start training from score -5.894136
[LightGBM] [Info] Start training from score -2.528510
[LightGBM] [Info] Start training from score -1.145809
[LightGBM] [Info] Start training from score -6.725434
[LightGBM] [Info] Start training from score -0.513663


,boosting_type,'gbdt'
,num_leaves,26
,max_depth,12
,learning_rate,0.005082590397179811
,n_estimators,479
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [20]:
model.score(X_test_final, y_test_enc)

0.8744